# Этап 1: Проверка модели на одной галактике
   
   Цель: подогнать модель и проверить качество подгонки.

In [ ]:
# %% [markdown]
# # Этап 1 (v2): Уточнённая модель упругого пространства
# 
# Проблема v1: Константа v0 портила внутреннюю часть кривой.
# Решение: Проверяем две гипотезы:
# 1. Подгонка v0 только по внешней половине галактики.
# 2. Плавное включение упругого члена: v^2 = v_vis^2 + v0^2 * (1 - exp(-r/r_t))

# %%
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import sys
import os

# Переход в корень проекта
current_dir = os.path.dirname(os.path.abspath('__file__'))
if current_dir.endswith('notebooks'):
    os.chdir('..')
sys.path.insert(0, os.path.abspath('src'))
from data_loader import load_rotation_curve, list_galaxies

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# %% [markdown]
# ## Загрузка данных (NGC 3198)

# %%
galaxies = list_galaxies()
galaxy_name = next((g for g in galaxies if "NGC3198" in g.upper()), galaxies[20])
print(f"🔬 Анализируем: {galaxy_name}")

data = load_rotation_curve(galaxy_name)
r = data['r']
v_obs = data['v_obs']
v_err = data['v_err']
v_vis = data['v_visible']

# %% [markdown]
# ## Модель 1: Подгонка только по внешней половине (r > r_mid)

# %%
r_mid = np.median(r)
mask_outer = r >= r_mid

def v_model_const(v_vis, v0):
    return np.sqrt(v_vis**2 + v0**2)

# Подгоняем ТОЛЬКО по внешним точкам
try:
    v0_guess = np.sqrt(max(0, v_obs[mask_outer][-1]**2 - v_vis[mask_outer][-1]**2)) + 20
    popt1, _ = curve_fit(v_model_const, v_vis[mask_outer], v_obs[mask_outer], p0=v0_guess)
    v0_outer = popt1[0]
    
    # Считаем chi2 для ВСЕЙ кривой с этим v0
    v_pred1 = v_model_const(v_vis, v0_outer)
    chi2_1 = np.sum(((v_obs - v_pred1) / v_err)**2)
    chi2_1_red = chi2_1 / (len(r) - 1)
except Exception as e:
    v0_outer = 0
    chi2_1_red = 999
    print(f"Ошибка подгонки модели 1: {e}")

print(f"\n📊 Модель 1 (v0 только для r > {r_mid:.1f} кпк):")
print(f"   v0 = {v0_outer:.1f} км/с")
print(f"   χ²/ndof (по всем точкам) = {chi2_1_red:.2f}")

# %% [markdown]
# ## Модель 2: Плавное включение упругости

# %%
def v_model_smooth(r, v_vis, v0, rt):
    """rt - радиус перехода, фиксируем его для простоты, например, на 1/3 макс радиуса"""
    transition = 1 - np.exp(-r / rt)
    return np.sqrt(v_vis**2 + (v0 * np.sqrt(transition))**2)

# Фиксируем rt = 5 кпк (характерный масштаб диска), подгоняем только v0
rt_fixed = 5.0 

def fit_func(r, v0):
    return v_model_smooth(r, v_vis, v0, rt_fixed)

try:
    popt2, _ = curve_fit(fit_func, r, v_obs, p0=[50], sigma=v_err)
    v0_smooth = popt2[0]
    v_pred2 = fit_func(r, v0_smooth)
    chi2_2 = np.sum(((v_obs - v_pred2) / v_err)**2)
    chi2_2_red = chi2_2 / (len(r) - 1)
except Exception as e:
    v0_smooth = 0
    chi2_2_red = 999

print(f"\n📊 Модель 2 (плавное включение, rt = {rt_fixed} кпк):")
print(f"   v0 = {v0_smooth:.1f} км/с")
print(f"   χ²/ndof (по всем точкам) = {chi2_2_red:.2f}")

# %% [markdown]
# ## Визуальное сравнение

# %%
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# График 1: Кривые вращения
ax1 = axes[0]
ax1.plot(r, v_obs, 'ko', label='Наблюдения (SPARC)', markersize=5, zorder=5)
ax1.plot(r, v_vis, 'g--', linewidth=1.5, label='Только видимое вещество', zorder=2)
ax1.plot(r, v_pred1, 'r-', linewidth=2, label=f'Модель 1 (внешняя подгонка, $v_0={v0_outer:.0f}$)', zorder=3)
ax1.plot(r, v_pred2, 'b-', linewidth=2, label=f'Модель 2 (плавная, $v_0={v0_smooth:.0f}$)', zorder=4)

ax1.set_xlabel('Радиус (кпк)')
ax1.set_ylabel('Скорость вращения (км/с)')
ax1.set_title(f'{galaxy_name}: Сравнение моделей упругого пространства')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axvline(r_mid, color='gray', linestyle=':', label=f'Граница подгонки (r={r_mid:.1f})')

# График 2: Остатки (Residuals)
ax2 = axes[1]
res1 = (v_obs - v_pred1) / v_err
res2 = (v_obs - v_pred2) / v_err

ax2.plot(r, res1, 'rs', markersize=4, label=f'Остатки Модели 1 ($\chi^2_r$={chi2_1_red:.1f})')
ax2.plot(r, res2, 'b^', markersize=4, label=f'Остатки Модели 2 ($\chi^2_r$={chi2_2_red:.1f})')
ax2.axhline(0, color='black', linewidth=1)
ax2.set_xlabel('Радиус (кпк)')
ax2.set_ylabel('Остатки ($\sigma$)')
ax2.set_title('Отклонения модели от наблюдений (в единицах погрешности)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-5, 5) # Ограничим масштаб, чтобы видеть разброс

plt.tight_layout()

save_dir = os.path.abspath('results/figures')
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f'{galaxy_name}_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 График сравнения сохранён в: {save_path}")

In [ ]:
# %% [markdown]
# # Тест на галактике, где видимого вещества МАЛО (UGC 128)
# Здесь наша модель упругости должна показать себя во всей красе.

# %%
# Ищем UGC 128 (иногда в базе она UGC0128 или UGC128)
target_galaxy = next((g for g in galaxies if "UGC128" in g.upper().replace("0", "")), None)
if target_galaxy is None:
    target_galaxy = "DDO154" # Запасной вариант, карликовая галактика

print(f"🔬 Переключаемся на галактику с дефицитом видимой массы: {target_galaxy}")

data2 = load_rotation_curve(target_galaxy)
r2 = data2['r']
v_obs2 = data2['v_obs']
v_err2 = data2['v_err']
v_vis2 = data2['v_visible']

print(f"Диапазон скоростей: наблюд. {v_obs2.max():.1f} км/с, видимое {v_vis2.max():.1f} км/с")
print(f"Дефицит скорости на краю: {v_obs2[-1]:.1f} - {v_vis2[-1]:.1f} = {v_obs2[-1] - v_vis2[-1]:.1f} км/с")

# %% [markdown]
# ## Применяем Модель 2 (Плавное включение)

# %%
rt_fixed = 3.0 # Для меньших галактик радиус перехода меньше

def fit_func_2(r, v0):
    transition = 1 - np.exp(-r / rt_fixed)
    return np.sqrt(v_vis2**2 + (v0 * np.sqrt(transition))**2)

try:
    # Начальное приближение: разница на последнем радиусе
    v0_guess_2 = max(10.0, np.sqrt(max(0, v_obs2[-1]**2 - v_vis2[-1]**2)))
    popt_2, _ = curve_fit(fit_func_2, r2, v_obs2, p0=[v0_guess_2], sigma=v_err2)
    v0_smooth_2 = popt_2[0]
    
    v_pred_2 = fit_func_2(r2, v0_smooth_2)
    chi2_2_new = np.sum(((v_obs2 - v_pred_2) / v_err2)**2)
    chi2_2_red_new = chi2_2_new / (len(r2) - 1)
    
    print(f"\n✅ Результаты для {target_galaxy}:")
    print(f"   v0 = {v0_smooth_2:.1f} км/с")
    print(f"   χ²/ndof = {chi2_2_red_new:.2f}")
    
    if chi2_2_red_new < 3.0:
        print("   🎉 ОТЛИЧНО! Модель прекрасно описывает данные там, где видимого вещества не хватает!")
    else:
        print("   ⚠️ Модель улучшила ситуацию, но требует дальнейшей доработки.")
        
except Exception as e:
    print(f"Ошибка подгонки: {e}")

# %% [markdown]
# ## Визуализация для UGC 128 / DDO 154

# %%
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(r2, v_obs2, 'ko', label='Наблюдения', markersize=6, zorder=5)
ax.plot(r2, v_vis2, 'g--', linewidth=2, label='Видимое вещество (звёзды+газ)', zorder=2)
ax.plot(r2, v_pred_2, 'b-', linewidth=2.5, label=rf'Модель упругости ($v_0={v0_smooth_2:.0f}$ км/с)', zorder=3)

ax.set_xlabel('Радиус (кпк)', fontsize=12)
ax.set_ylabel('Скорость вращения (км/с)', fontsize=12)
ax.set_title(f'{target_galaxy}', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_path_2 = os.path.join(os.path.abspath('results/figures'), f'{target_galaxy}_elastic_success.png')
plt.savefig(save_path_2, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 График сохранён в: {save_path_2}")

In [ ]:
# %% [markdown]
# # Этап 1 (v3): Двухпараметрическая подгонка для UGC 128
# Теперь мы позволяем алгоритму найти не только v0, но и идеальный радиус перехода rt.

# %%
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import sys
import os

# Убеждаемся, что мы в корне
current_dir = os.path.dirname(os.path.abspath('__file__'))
if current_dir.endswith('notebooks'):
    os.chdir('..')
sys.path.insert(0, os.path.abspath('src'))
from data_loader import load_rotation_curve, list_galaxies

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# %%
galaxy_name = "UGC00128_rotmod"
print(f"🔬 Глубокий анализ галактики: {galaxy_name}")

data = load_rotation_curve(galaxy_name)
r = data['r']
v_obs = data['v_obs']
v_err = data['v_err']
v_vis = data['v_visible']

# %% [markdown]
# ## Модель с двумя свободными параметрами: v0 и rt

# %%
def v_model_2param(r, v0, rt):
    """
    v0: характеристическая скорость упругого провиса
    rt: радиус, на котором упругий эффект начинает плавно включаться
    """
    # Защита от деления на ноль или отрицательных rt в процессе подгонки
    rt = max(rt, 0.1) 
    transition = 1 - np.exp(-r / rt)
    return np.sqrt(v_vis**2 + (v0 * np.sqrt(transition))**2)

# Начальные предположения:
# v0_guess: разница скоростей на последнем радиусе
v0_guess = max(10.0, np.sqrt(max(0, v_obs[-1]**2 - v_vis[-1]**2)))
# rt_guess: примерно 1/4 от максимального радиуса галактики
rt_guess = r.max() / 4.0

print(f"Начальные предположения: v0 = {v0_guess:.1f}, rt = {rt_guess:.1f}")

# Подгонка с ограничениями (bounds), чтобы параметры не ушли в отрицательные значения
# bounds=([0, 0.1], [300, 20]) означает: v0 от 0 до 300, rt от 0.1 до 20 кпк
try:
    popt, pcov = curve_fit(
        v_model_2param, r, v_obs, 
        p0=[v0_guess, rt_guess], 
        sigma=v_err,
        bounds=([0, 0.1], [300, 20])
    )
    
    v0_best = popt[0]
    rt_best = popt[1]
    v0_err = np.sqrt(pcov[0, 0])
    rt_err = np.sqrt(pcov[1, 1])
    
    v_pred = v_model_2param(r, v0_best, rt_best)
    
    chi2 = np.sum(((v_obs - v_pred) / v_err)**2)
    ndof = len(r) - 2  # Вычитаем 2, так как у нас теперь ДВА свободных параметра
    chi2_red = chi2 / ndof
    
    print(f"\n🎉 ОПТИМАЛЬНЫЕ ПАРАМЕТРЫ:")
    print(f"   v0 = {v0_best:.1f} ± {v0_err:.1f} км/с")
    print(f"   rt = {rt_best:.2f} ± {rt_err:.2f} кпк")
    print(f"   χ²/ndof = {chi2_red:.2f}")
    
    if chi2_red < 5.0:
        print("   ✅ Отличное количественное согласие модели с данными!")
    else:
        print("   ⚠️ Модель описывает тренд, но есть систематические отклонения.")

except Exception as e:
    print(f"❌ Ошибка подгонки: {e}")
    v0_best, rt_best = v0_guess, rt_guess
    v_pred = v_vis # fallback

# %% [markdown]
# ## Визуализация лучшего совпадения

# %%
fig, ax = plt.subplots(figsize=(10, 6))

# Наблюдения с error bars
ax.errorbar(r, v_obs, yerr=v_err, fmt='ko', markersize=5, label='Наблюдения (SPARC)', zorder=5, alpha=0.8)

# Компоненты
ax.plot(r, v_vis, 'g--', linewidth=2, label='Видимое вещество (звёзды + газ)', zorder=2)
ax.plot(r, v_pred, 'b-', linewidth=2.5, label=rf'Упругая модель ($v_0={v0_best:.0f}$, $r_t={rt_best:.1f}$ кпк)', zorder=3)

# Аннотация с результатами прямо на графике
textstr = '\n'.join((
    rf'$v_0 = {v0_best:.1f} \pm {v0_err:.1f}$ км/с',
    rf'$r_t = {rt_best:.2f} \pm {rt_err:.2f}$ кпк',
    rf'$\chi^2/\nu = {chi2_red:.2f}$'
))
props = dict(boxstyle='round', facecolor='white', alpha=0.8)
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

ax.set_xlabel('Радиус (кпк)', fontsize=12)
ax.set_ylabel('Скорость вращения (км/с)', fontsize=12)
ax.set_title(f'{galaxy_name}: Двухпараметрическая подгонка упругой модели', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_dir = os.path.abspath('results/figures')
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f'{galaxy_name}_best_fit.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 Лучший график сохранён в: {save_path}")